# 00 — Profiling the source files

**Goal:** find out what is wrong with the data *before* writing any matching code.

Nobody hands you a list of data problems on a real project — you find them yourself.
Every finding here is written up in `docs/DATA_ISSUES.md` with an ID (D-001, D-002 …),
and each section below says which issues it produced.

Three parts:
1. **Overview** of every file — size, gaps, lengths
2. **Names** — how each system writes customer names
3. **Suites** — where the suite number hides

## Setup

**What the code does**

- `Path("../data/sample")` — the folder with the CSV files. `..` means "one folder up",
  because this notebook sits inside `notebooks/`.
- `load(name)` — a shortcut so we can write `load("erp_customer_master")` instead of the
  full `read_csv` line every time.
- **`dtype=str` is important.** It reads every column as text. Without it, pandas would
  turn the ZIP code `01234` into the number `1234` and silently lose the leading zero.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

path = Path("../data/sample")

def load(name):
    # Read one CSV, keeping every column as text
    return pd.read_csv(path / f"{name}.csv", dtype=str)

## Part 1 — Overview of every file

**Why:** a quick health check of each file, to see where to dig deeper.

**What the code does**

`profile_file` prints one header line about the whole file, then returns a small table
with **one row per column**:

| Column | Meaning | What it tells you |
|---|---|---|
| `distinct` | how many different values | equal to the row count → an ID; very few → a category |
| `nulls` | how many empty cells | a column that is mostly empty |
| `min_len` / `max_len` | shortest and longest value | a suspicious round `max_len` (like 30) → text was cut off |

How the code works, line by line:

- `df.duplicated(subset=...)` — marks a row as a duplicate if the chosen columns repeat an
  earlier row. `.sum()` counts them (True counts as 1).
- `df.apply(lambda col: col.str.len())` — runs `.str.len()` on **each column**, giving the
  length of every value. Empty cells stay empty and are ignored by `min` / `max`.
- `pd.DataFrame({...})` — builds the result table. Each dictionary key becomes a column.

The `sources` list says, for each file, **which columns together identify a customer** —
those are the columns used for the duplicate check.

**Produced:** D-001 (truncation), D-005 (mixed ZIP), D-006 (names repeat), D-007 (ERP duplicates), D-008 (NPI gaps)

In [ ]:
def profile_file(df, name, dup_subset=None):
    # Facts about the whole file, printed once
    dups = df.duplicated(dup_subset).sum()
    print(f"=== {name}: {len(df)} rows, {len(df.columns)} columns, {dups} duplicates on {dup_subset} ===")

    # Length of every value in every column
    lengths = df.apply(lambda col: col.str.len())

    # One row per column
    return pd.DataFrame({
        "distinct": df.nunique(),
        "nulls":    df.isna().sum(),
        "min_len":  lengths.min(),
        "max_len":  lengths.max(),
    })


# For each file: the columns that together identify a customer
sources = [
    ("erp_customer_master",                  ["customer_name", "postal_code"]),
    ("salesforce_accounts",                  ["account_name", "billing_zip"]),
    ("provider_reference",                   ["organization_name", "zip5"]),
    ("distributor_alpha_tracing_2026-03",    ["ship_to_name", "ship_to_zip"]),
    ("distributor_beta_tracing_2026-03",     ["CustomerName", "CityStateZip"]),
    ("distributor_gamma_tracing_2026-04-01", ["ship_to_name", "ship_to_zip"]),
    ("gpo_roster",                           ["gpo_id", "npi"]),
    ("legacy_match_decisions",               ["source_system", "source_record_id"]),
]

for name, subset in sources:
    display(profile_file(load(name), name, subset))

=== erp_customer_master: 63 rows, 9 columns, 3 duplicates on ['customer_name', 'postal_code'] ===


,distinct,nulls,min_len,max_len
erp_customer_id,63,0,5.0,5.0
customer_name,53,0,11.0,30.0
address_line_1,52,0,9.0,24.0
address_line_2,7,48,6.0,18.0
address_line_3,4,59,8.0,9.0
city,9,0,5.0,11.0
state,6,0,2.0,2.0
postal_code,9,0,5.0,5.0
legacy_customer_id,61,0,6.0,6.0


=== salesforce_accounts: 62 rows, 8 columns, 0 duplicates on ['account_name', 'billing_zip'] ===


,distinct,nulls,min_len,max_len
sf_account_id,62,0,6.0,6.0
account_name,55,0,16.0,45.0
billing_street,52,0,11.0,28.0
billing_suite,8,54,8.0,9.0
billing_city,9,0,5.0,11.0
billing_state,6,0,2.0,2.0
billing_zip,36,0,5.0,10.0
npi_number,45,17,10.0,10.0


=== provider_reference: 61 rows, 11 columns, 1 duplicates on ['organization_name', 'zip5'] ===


...

,distinct,nulls,min_len,max_len
trace_id,27,0,8.0,8.0
invoice_date,1,0,10.0,10.0
ship_to_name,27,0,14.0,43.0
ship_to_address,23,4,7.0,26.0
ship_to_city,14,0,5.0,11.0
ship_to_state,6,0,2.0,2.0
ship_to_zip,11,1,5.0,5.0
product_code,5,0,8.0,8.0
quantity,17,0,1.0,2.0
net_sales_amount,27,0,6.0,7.0


=== gpo_roster: 62 rows, 4 columns, 0 duplicates on ['gpo_id', 'npi'] ===


,distinct,nulls,min_len,max_len
gpo_id,3,0,6,6
gpo_name,3,0,17,20
npi,44,0,10,10
member_since,48,0,10,10


=== legacy_match_decisions: 220 rows, 7 columns, 0 duplicates on ['source_system', 'source_record_id'] ===


,distinct,nulls,min_len,max_len
decision_id,220,0,6.0,6.0
source_system,2,0,16.0,17.0
source_record_id,220,0,7.0,8.0
legacy_customer_id,59,18,6.0,6.0
match_level,6,0,8.0,21.0
decided_by,4,0,6.0,7.0
decided_on,9,0,10.0,10.0


## Part 2 — How each system writes customer names

**Why:** the same hospital is written differently in each system (`HOSP` vs `Hospital`).
To match them later, we need the full list of these differences.

The method is to break every name into single words — **tokens** — and count them.

**What the code does**

`clean_words` turns each name into a list of words, in three steps:

| Step | Code | `"Mercy Hospital, Inc."` becomes |
|---|---|---|
| 1. upper-case | `.str.upper()` | `"MERCY HOSPITAL, INC."` |
| 2. punctuation → space | `.str.replace(r"[^A-Z0-9 ]", " ", regex=True)` | `"MERCY HOSPITAL  INC "` |
| 3. split into words | `.str.split()` | `["MERCY", "HOSPITAL", "INC"]` |

The pattern `[^A-Z0-9 ]` means *"any character that is NOT a letter, digit or space"*.
The `^` inside the brackets means NOT.

**Why step 2 matters:** without it, `Hospital,` (with the comma) counts as a different word
from `Hospital`. That is exactly why Excel showed 20 hospitals while the first version of
this code showed 19.

`tokens` then:
- `.explode()` — each word gets its **own row** (one row of 3 words becomes 3 rows)
- `.value_counts()` — counts how often each word appears

**Produced:** D-010 (punctuation)

In [ ]:
erp   = load("erp_customer_master")
sf    = load("salesforce_accounts")
alpha = load("distributor_alpha_tracing_2026-03")

def clean_words(names):
    # Upper-case, punctuation -> space, split into a list of words
    return names.str.upper().str.replace(r"[^A-Z0-9 ]", " ", regex=True).str.split()

def tokens(names):
    # One row per word, then count each word
    return clean_words(names).explode().value_counts()

### First and last words

**Why:** two problems hide at the edges of a name, and they need different fixes.

- **First word** — is it `ST`? At the start of a name, `ST` means *Saint*. In an address
  it means *Street*. The fix has to depend on the position.
- **Last word** — is it `INC`, `LLC`, `PC`? These legal endings identify nothing and should
  be removed before comparing names.

**What the code does**

`clean_words(...)` gives a list of words per name. On a list:
- `.str[0]` takes the **first** word
- `.str[-1]` takes the **last** word (`-1` means "count from the end")

`.head(8)` shows only the 8 most common, to keep it short.

In [ ]:
for label, names in [("erp", erp.customer_name), ("salesforce", sf.account_name)]:
    words = clean_words(names)
    print(f"--- {label}: first words ---")
    display(words.str[0].value_counts().head(8))
    print(f"--- {label}: last words ---")
    display(words.str[-1].value_counts().head(8))

--- erp: first words ---


customer_name
ST            9
NORTHGATE     7
WILLAMETTE    7
LAKEVIEW      5
GRANITE       5
CEDAR         5
BROOKSIDE     4
SILVER        4
Name: count, dtype: int64

--- erp: last words ---


customer_name
CTR        22
HOSP       19
INST       11
ORTHO       7
CLNC        3
RESEARC     1
Name: count, dtype: int64

--- salesforce: first words ---


account_name
SAINT         9
LAKEVIEW      6
NORTHGATE     6
WILLAMETTE    6
GRANITE       5
SILVER        4
CEDAR         4
BROOKSIDE     3
Name: count, dtype: int64

--- salesforce: last words ---


account_name
CENTER         20
HOSPITAL       18
INSTITUTE       9
ORTHOPEDICS     5
LLC             5
INC             3
CLINIC          2
Name: count, dtype: int64

### Comparing two systems — finding the abbreviations

**Why:** a word that appears in one system but **never** in the other is usually an
abbreviation. `HOSP` is only in the ERP, `HOSPITAL` only in Salesforce — so they are a pair.

**What the code does**

`compare_tokens` works in three steps:

1. **`pd.concat({...}, axis=1)`** — puts the two word counts side by side, lined up by word.
   If a word is missing from one side, that cell is empty (`NaN`).
2. **`.isna()`** — keeps only the words where the *other* side is empty, i.e. words used by
   one system only.
3. **`.reset_index()` + `pd.concat`** — lines the two lists up by **position** (1st with 1st,
   2nd with 2nd) so the pairs sit next to each other.

**Read the result with your own judgement.** Both lists are sorted by count, so the top
rows usually pair up correctly (`CTR 22 ↔ CENTER 21`). Lower down they can be misaligned —
for example `CLNC` may sit next to `LLC`, when its real partner is `CLINIC`. The code only
lines them up; deciding which word means which is a human call.

**Produced:** D-002 / D-009 (the abbreviation dictionary), D-011 (different populations)

In [28]:
def compare_tokens(left, right, left_name, right_name):
    # 1. Word counts side by side, lined up by word
    both = pd.concat({left_name: tokens(left), right_name: tokens(right)}, axis=1)

    # 2. Words used by only one of the two systems
    left_only  = both[both[right_name].isna()][left_name].sort_values(ascending=False)
    right_only = both[both[left_name].isna()][right_name].sort_values(ascending=False)

    # 3. Line the two lists up by position so the pairs sit next to each other
    side = pd.concat([left_only.reset_index(), right_only.reset_index()], axis=1)
    side.columns = [f"{left_name}_word", f"{left_name}_n", f"{right_name}_word", f"{right_name}_n"]
    return side


compare_tokens(erp.customer_name, sf.account_name, "erp", "sf")

,erp_word,erp_n,sf_word,sf_n
0,CTR,22.0,CENTER,21.0
1,HOSP,19.0,HOSPITAL,20.0
2,INST,11.0,INSTITUTE,11.0
3,ST,9.0,SAINT,9.0
4,ORTHO,7.0,ORTHOPEDICS,7.0
5,MED,7.0,MEDICAL,6.0
6,CLNC,3.0,LLC,5.0
7,RESEARC,1.0,CLINIC,3.0
8,NaN,NaN,INC,3.0
9,NaN,NaN,CASCADE,1.0


### The distributor file — the dirty one

**Why:** the tracing files are typed by warehouse staff at another company, with no naming
rules. They are the records this whole project exists to match.

**What the code does:** the same function, now ERP against Distributor Alpha.

**Read the right-hand column.** The left column is almost empty — and that is itself a
finding. The distributor uses **both** forms, `CTR` *and* `CENTER`, `HOSP` *and* `HOSPITAL`,
sometimes in the same file. So the ERP's abbreviations are no longer unique to the ERP.
That means the cleaning rules have to be applied to every source, not just the ERP.

**What to look for:**
- `D` and `B` high up — the slashes in `D/B/A` (*doing business as*) became separate letters
- `P` and `C` — the same thing happening to `P.C.`
- stray numbers like `210`, `330` — suite numbers left inside the name

**Produced:** D-012

In [ ]:
compare_tokens(erp.customer_name, alpha.ship_to_name, "erp", "alpha").head(20)

,erp_word,erp_n,alpha_word,alpha_n
0,RESEARC,1.0,CENTER,38.0
1,NaN,NaN,HOSPITAL,37.0
2,NaN,NaN,D,26.0
3,NaN,NaN,B,26.0
4,NaN,NaN,HEALTH,26.0
5,NaN,NaN,INSTITUTE,25.0
6,NaN,NaN,SAINT,20.0
7,NaN,NaN,ORTHOPEDICS,18.0
8,NaN,NaN,P,11.0
9,NaN,NaN,C,11.0


## Part 3 — Where does the suite number live?

**Why:** two different businesses can share a street address and differ only by suite
(`Suite 400` vs `Suite 210`). To compare suites we first have to **find** them — and they
are not always in the same column.

### What is really in the ERP's second address line?

**What the code does**
- `.dropna()` — removes the empty cells
- `.value_counts()` — lists each distinct value and how often it appears

**What to look for:** only some values are suites (`STE 400`). Others are delivery notes
(`ATTN RECEIVING`, `LOADING DOCK B`). So we cannot just trust this column.

**Produced:** D-003

In [ ]:
erp.address_line_2.dropna().value_counts()

address_line_2
ATTN RECEIVING        5
C/O MATERIALS MGMT    3
LOADING DOCK B        3
STE 400               1
STE 210               1
STE 2B                1
STE 200               1
Name: count, dtype: int64

### Searching every column for a suite

**What the code does**

For each file and each column, it counts how many values contain a suite marker.

`.str.contains(SUITE, case=False, na=False, regex=True)` returns True/False for each row,
and `.sum()` counts the Trues.
- `case=False` — ignores upper/lower case
- `na=False` — treats empty cells as "no match" instead of an error

**The pattern, explained piece by piece:**

| Piece | Meaning |
|---|---|
| `STE\|SUITE\|UNIT\|RM` | any one of these words (`\|` means OR) |
| `(?: ... )` | groups them together, so `\b` applies to all of them |
| `\b` | a **word boundary** — the word must stand alone |
| `\|#` | or a `#` sign |

**Why `\b` is essential:** without it, `UNIT` matches inside `COMMUNITY`, `RM` inside `FARM`,
and `STE` inside `SYSTEM`. The first version of this search reported 8 suites in the ERP
name column — all of them were the word `COMMUNITY`. With `\b` the count is 0.

**What to look for:** suites in `ship_to_name`. That means the suite can be inside the
**name**, so the code that extracts suites later must check the name too, not only the
address.

**Produced:** D-004, D-012

In [ ]:
SUITE = r"\b(?:STE|SUITE|UNIT|RM)\b|#"

for label, df in [("erp", erp), ("salesforce", sf), ("alpha", alpha)]:
    for col in df.columns:
        hits = df[col].str.contains(SUITE, case=False, na=False, regex=True).sum()
        if hits > 0:
            print(f"{label:11} {col:18} {hits}")

erp         address_line_2     4
erp         address_line_3     4
salesforce  billing_suite      8
alpha       ship_to_name       4
alpha       ship_to_address    14


## Summary

12 issues found, written up in `docs/DATA_ISSUES.md`. Each one becomes a rule in
`src/mdm/standardize.py` (the next step, A.1).

The three that shape the design most:

- **D-009** — the abbreviation dictionary (`HOSP` → `HOSPITAL` …)
- **D-010** — the order of cleaning steps: punctuation must go *before* splitting into words
- **D-012** — the suite can be inside the name, so the suite finder has to look there too